In [55]:
import os
os.chdir("/Users/naginilandarhi/DSAICourse/Prof_Shankar_Venkatagiri/LinAlg4DataScience/faker_data_synthesizer")

In [56]:
import pandas as pd
import numpy as np

In [57]:
print(os.getcwd())

/Users/naginilandarhi/DSAICourse/Prof_Shankar_Venkatagiri/LinAlg4DataScience/faker_data_synthesizer


In [ ]:
r100000 = pd.read_csv('rows100000/synthetic_data_100000_rows_10_cols_17_clusters.csv')

In [59]:
r100000.head()

,ID,col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9
0,e0c53cb83da942a98ed42f1a3d4cbf37,810.183034,579.397842,846.655621,524.384309,682.181626,-50.588939,156.737651,224.434401,-8.905864
1,f6e07cc06c52449f9b49bd26df57c59a,863.232929,247.305112,625.719838,334.864908,903.397753,399.901305,203.207231,168.815325,514.103976
2,3985c3cf3f764e1d9efa21977394988f,864.747662,246.461925,616.059258,333.460198,912.141427,405.014599,192.732638,171.819347,520.945549
3,7c441fe7ab4240a7874a493b3ceddf2d,864.692430,246.610714,623.017144,339.879255,903.993274,397.973366,192.541145,172.190223,515.123126
4,6c6fa6115ab34edfae595ed3a8b317fa,-78.197849,147.360430,464.784501,-67.660730,117.330225,613.051652,497.847504,130.955612,551.461374


In [60]:
r100000.dtypes

ID        object
col_1    float64
col_2    float64
col_3    float64
col_4    float64
col_5    float64
col_6    float64
col_7    float64
col_8    float64
col_9    float64
dtype: object

In [61]:
data_matrix_a = r100000.drop(columns=['ID']).to_numpy(dtype=np.float32)

In [62]:
data_matrix_a.shape

(100000, 9)

In [63]:
ids = r100000["ID"].astype(str).to_numpy()

I need to do is to deal with id column. I will drop the id column from the data matrix. I will keep only the feature columns in the data matrix.

Now I need to identify which cluster each row belongs to. I do not want to use KMeans from sklearn. I want to implement KMeans from scratch. I will use the following steps:
1. Randomly select k rows from the data matrix as the initial centroids.
2. Assign each row to the nearest centroid based on Euclidean distance.
3. Recalculate the centroids as the mean of the rows assigned to each centroid.
4. Repeat steps 2 and 3 until convergence (i.e., the centroids do not change significantly between iterations). 

In [64]:
import time

In [65]:
def kmeans_cpu(X,
               K,
               initial_centroids=None,
               max_iters=100,
               tol=1e-4,
               seed=42):
    N, D = X.shape
    if initial_centroids is None:
        rng = np.random.default_rng(seed)
        indices = rng.choice(N, K, replace=False)
        centroids = X[indices].copy()

    else:
        centroids = initial_centroids.copy()
        centroids = centroids.astype(np.float32)

    assignments = np.full(N, -1, dtype=np.int32)   
    start_time = time.perf_counter()

    for iteration in range(max_iters):
        distances = np.linalg.norm(X[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
        new_assignments = np.argmin(distances, axis=1)

        if np.array_equal(assignments, new_assignments):
            print(f"Converged after {iteration} iterations.")
            break

        assignments = new_assignments
        new_centroids = np.empty_like(centroids)

        for k in range(K):
            assigned_points = X[assignments == k]
            if len(assigned_points) > 0:
                new_centroids[k] = assigned_points.mean(axis=0)
                new_centroids[k] = new_centroids[k].astype(np.float32)

        if np.array_equal(centroids, new_centroids):
            print(f"Converged after {iteration} iterations.")
            break

        movement = np.linalg.norm(
            new_centroids - centroids,
            axis=1
        )

        max_movement = np.max(movement)
        centroids = new_centroids
        if max_movement < tol:
            break

    end_time = time.perf_counter()
    execution_time = end_time - start_time
    return {
        "assignments": assignments,
            "centroids": centroids,
            "execution_time": execution_time,
            "iterations": iteration + 1,
            "max_movement": max_movement
        }        


In [ ]:
rng = np.random.default_rng(42)
initial_centroids = rng.choice(len(data_matrix_a), size=17, replace=False)
initial_centroids = data_matrix_a[initial_centroids]
print("Initial centroids:\n", initial_centroids)

Initial centroids:
 [[ 2.5975115e+01  5.9158844e+02  7.5657306e+02  3.6882117e+02
  -1.7582853e+01  3.3490430e+02  9.9968408e+02  4.9400925e+02
   9.7856073e+02]
 [ 5.9374072e+02 -6.3316067e+01  1.9869125e+02  1.4112688e+02
   7.0526166e+02  6.4956274e+02  8.7789203e+02 -4.1972470e+00
   3.6857224e+02]
 [ 5.9869971e+02 -6.7418137e+01  2.1651888e+02  1.4927486e+02
   7.0907874e+02  6.4333215e+02  8.8073407e+02  7.8355783e-01
   3.7360269e+02]
 [ 1.8685623e+02  5.4059082e+02  8.8814197e+02  3.3871524e+02
   1.4090413e+02  9.9653284e+02  4.6154538e+02  1.0586883e+00
  -4.7610455e+01]
 [ 8.0975610e+02  5.7066504e+02  8.5238354e+02  5.4600537e+02
   6.7336359e+02 -4.8859180e+01  1.4384514e+02  2.1782204e+02
  -1.0284873e+01]
 [ 6.0259668e+02 -7.7158455e+01  2.0415349e+02  1.3992838e+02
   7.1766650e+02  6.4513849e+02  8.7523840e+02 -1.3707537e+01
   3.5870175e+02]
 [ 1.5811217e+01  8.2872101e+02  5.7288855e+02  8.0660229e+02
   7.0014276e+02  4.7966449e+02  9.8215015e+02  3.2004099e+02
   5

In [ ]:
result = kmeans_cpu(data_matrix_a, K=17, initial_centroids=initial_centroids, max_iters=1000, tol=1e-4, seed=42)
print(f"Execution time: {result['execution_time']:.4f} seconds")
print(f"Iterations: {result['iterations']}")
print(f"Max movement: {result['max_movement']:.6f}")
print(f"Centroids:\n{result['centroids']}")

Converged after 177 iterations.
Execution time: 2.6085 seconds
Iterations: 178
Max movement: 0.008305
Centroids:
[[ 1.81366711e+01  5.90500610e+02  7.71734009e+02  3.63932617e+02
  -3.00519123e+01  3.20033325e+02  1.00068768e+03  4.87205811e+02
   9.69529297e+02]
 [ 6.03413696e+02 -6.97315826e+01  2.03284607e+02  1.42358795e+02
   7.06349243e+02  6.47081360e+02  8.81278015e+02 -1.28013754e+00
   3.65160706e+02]
 [ 6.00616333e+02 -7.32718430e+01  2.01853256e+02  1.49198257e+02
   7.14517578e+02  6.44750977e+02  8.83432373e+02 -3.67302704e+00
   3.66551941e+02]
 [-6.72048798e+01  1.40630722e+02  4.55833588e+02 -7.08240280e+01
   1.18665123e+02  6.14814819e+02  4.99349274e+02  1.42508255e+02
   5.48193481e+02]
 [ 8.38216248e+02  4.13068634e+02  7.34264099e+02  4.34981964e+02
   7.90713318e+02  1.77892990e+02  1.71096130e+02  1.94755234e+02
   2.53025848e+02]
 [ 6.06179932e+02 -7.46362152e+01  2.02549393e+02  1.45053528e+02
   7.09972046e+02  6.41407166e+02  8.78883972e+02 -7.83746195e+00


In [68]:
output = pd.DataFrame({
    "ID": ids,
    "Cluster": result["assignments"]
})

output.to_csv("kmeans_output.csv", index=False)

In [69]:
np.savetxt(
    "initial_centroids.csv",
    initial_centroids,
    delimiter=","
)

In [70]:
def representative_points(
    X,
    ids,
    centroids,
    assignments
):
    representatives = []

    for k in range(len(centroids)):

        indices = np.where(assignments == k)[0]

        cluster_points = X[indices]

        diff = cluster_points - centroids[k]

        distances = np.sum(
            diff * diff,
            axis=1
        )

        nearest_local_index = np.argmin(distances)

        original_index = indices[nearest_local_index]

        representatives.append({
            "cluster": k,
            "ID": ids[original_index],
            "distance_to_centroid":
                float(np.sqrt(distances[nearest_local_index]))
        })

    return representatives

In [71]:
representatives = representative_points(
    data_matrix_a,
    ids,
    result["centroids"],
    result["assignments"]
)

print(representatives)

[{'cluster': 0, 'ID': '97865b6783714d839fc4904539dc77dd', 'distance_to_centroid': 6.837813854217529}, {'cluster': 1, 'ID': 'cd3a24c7afd34ae1becfbfc97088dc6d', 'distance_to_centroid': 3.8051204681396484}, {'cluster': 2, 'ID': 'c816b4343fe44f8789dc71077925619d', 'distance_to_centroid': 6.094925403594971}, {'cluster': 3, 'ID': 'e64c76254c634055bb4fdd7530922816', 'distance_to_centroid': 5.6438751220703125}, {'cluster': 4, 'ID': '58935e0c340d4c3ab044cd4e1c0b941c', 'distance_to_centroid': 410.7659606933594}, {'cluster': 5, 'ID': 'bfec9047c5af4833a653c56f626fd960', 'distance_to_centroid': 5.985167980194092}, {'cluster': 6, 'ID': 'a95a0a26eee0444493a24eb597ea0500', 'distance_to_centroid': 6.832719802856445}, {'cluster': 7, 'ID': 'f7ca50d9cff1415d99418be648c32e70', 'distance_to_centroid': 7.9478759765625}, {'cluster': 8, 'ID': 'f1b462d7ed8541e4aff956cf31da0016', 'distance_to_centroid': 279.7723388671875}, {'cluster': 9, 'ID': '0b58cb9e82d2471aac20f1e9d7028d84', 'distance_to_centroid': 5.5003409